In [39]:
#general setup

from pathlib import Path
from typing import Optional
from collections import Counter
import json
import re

import pandas as pd
import numpy as np


# -----------------------------
# Basic utilities
# -----------------------------

def find_repo_root(start: Optional[Path] = None, repo_name: str = "masters_thesis_sdg") -> Path:
    """
    Walk upward from the current working directory until the repository root is found.
    This makes the notebook robust even if it is launched from a subfolder.
    """
    if start is None:
        start = Path.cwd().resolve()

    current = start

    while True:
        if current.name == repo_name:
            return current

        if current.parent == current:
            raise FileNotFoundError(
                f"Could not find repo root '{repo_name}' starting from {start}"
            )

        current = current.parent


def load_json(path: Path):
    """
    Load a JSON file with UTF-8 encoding.
    """
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# -----------------------------
# Project paths
# -----------------------------

REPO_ROOT = find_repo_root()

IDX_FILE_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "lai2023"
    / "onuw_transcripts_ready"
    / "index_cleaned.json"
)

RESULTS_DIR = REPO_ROOT / "results" / "voting"

OUT_DIR = REPO_ROOT / "results" / "analysis" / "llm_vote_stability"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------
# Analysis configuration
# -----------------------------

llm = "unsloth_gemma-4-31B-it"
prompt_family = "v4"


# -----------------------------
# Load index
# -----------------------------

index_data = load_json(IDX_FILE_PATH)


print("REPO_ROOT:", REPO_ROOT)
print("IDX_FILE_PATH:", IDX_FILE_PATH)
print("RESULTS_DIR:", RESULTS_DIR)
print("OUT_DIR:", OUT_DIR)
print()
print("Loaded games from index:", len(index_data))
print("LLM:", llm)
print("Prompt family:", prompt_family)

REPO_ROOT: C:\Users\annab\Documents\GitHub\masters_thesis_sdg
IDX_FILE_PATH: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\processed\lai2023\onuw_transcripts_ready\index_cleaned.json
RESULTS_DIR: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\results\voting
OUT_DIR: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\results\analysis\llm_vote_stability

Loaded games from index: 191
LLM: unsloth_gemma-4-31B-it
Prompt family: v4


In [40]:
# -----------------------------
# Build clean ground-truth table from index_cleaned.json
# -----------------------------

def normalize_missing_vote(v):
    """
    Returns True if a vote value should be treated as missing.
    """
    if v is None:
        return True

    if isinstance(v, str):
        return v.strip().upper() in {"", "NA", "N/A", "NONE", "NULL"}

    return False


def clean_human_votes(voting_outcome):
    """
    Convert human voting outcome to a list of valid integer votes.
    
    The index sometimes contains values such as "N/A".
    We ignore them for descriptive human-vote analysis.
    """
    if voting_outcome is None:
        return []

    clean_votes = []

    for v in voting_outcome:
        if normalize_missing_vote(v):
            continue

        try:
            clean_votes.append(int(v))
        except Exception:
            continue

    return clean_votes


def summarize_human_vote(voting_outcome, end_roles):
    """
    Summarize the human voting outcome.

    Important:
    This is used only for descriptive analysis of human voting.
    The LLM correctness will be computed from final roles:
        - player vote is correct if the selected player is a final Werewolf
        - circle vote is correct if there is no final Werewolf
    """
    clean_votes = clean_human_votes(voting_outcome)
    n_missing_votes = 0 if voting_outcome is None else len(voting_outcome) - len(clean_votes)

    if len(clean_votes) == 0:
        return {
            "human_vote_available": False,
            "human_vote_type": None,
            "human_target_indices": [],
            "human_target_names": [],
            "human_vote_counts": {},
            "n_missing_human_votes": n_missing_votes,
        }

    vote_counts = Counter(clean_votes)
    max_votes = max(vote_counts.values())

    # In ONUW, if nobody receives at least two votes, nobody is eliminated.
    # We treat this as a circle/no-elimination outcome.
    if max_votes < 2:
        return {
            "human_vote_available": True,
            "human_vote_type": "circle",
            "human_target_indices": [],
            "human_target_names": [],
            "human_vote_counts": dict(vote_counts),
            "n_missing_human_votes": n_missing_votes,
        }

    target_indices = [
        idx
        for idx, count in vote_counts.items()
        if count == max_votes
    ]

    return {
        "human_vote_available": True,
        "human_vote_type": "player",
        "human_target_indices": target_indices,
        "human_target_names": [],  # filled later when player_names are available
        "human_vote_counts": dict(vote_counts),
        "n_missing_human_votes": n_missing_votes,
    }


def build_gt_df(index_data):
    """
    Build one clean row per game.

    The central identifier is game_name, derived from processed_txt_path.
    Example:
        Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#9_Game1

    This should match the stem of the result JSON filenames.
    """
    rows = []

    for game in index_data:
        source = game["source"]
        session_name = game["session_name"]
        game_key = game["game_key"]

        processed_txt_path = game.get("processed_txt_path")
        game_name = Path(processed_txt_path).stem if processed_txt_path else f"{session_name}_{game_key}"

        player_names = game["player_names"]
        start_roles = game["start_roles"]
        end_roles = game["end_roles"]
        voting_outcome = game.get("voting_outcome", [])

        correct_player_indices = [
            i
            for i, role in enumerate(end_roles)
            if role == "Werewolf"
        ]

        correct_player_names = [
            player_names[i]
            for i in correct_player_indices
        ]

        has_werewolf = len(correct_player_indices) > 0

        human_summary = summarize_human_vote(voting_outcome, end_roles)

        human_target_names = [
            player_names[i]
            for i in human_summary["human_target_indices"]
            if 0 <= i < len(player_names)
        ]

        rows.append({
            "game_name": game_name,
            "source": source,
            "session_name": session_name,
            "game_key": game_key,

            "processed_txt_path": processed_txt_path,

            "num_players": game.get("num_players"),
            "num_turns": game.get("num_turns"),
            "num_unparsed_lines": game.get("num_unparsed_lines"),
            "num_dropped_lines": game.get("num_dropped_lines"),

            "player_names": player_names,
            "start_roles": start_roles,
            "end_roles": end_roles,

            "has_werewolf": has_werewolf,
            "n_werewolves": len(correct_player_indices),
            "correct_player_indices": correct_player_indices,
            "correct_player_names": correct_player_names,

            # Correct target type for the LLM task
            "correct_vote_type": "player" if has_werewolf else "circle",

            # Human vote info, useful later for comparison
            "voting_outcome": voting_outcome,
            "human_vote_available": human_summary["human_vote_available"],
            "human_vote_type": human_summary["human_vote_type"],
            "human_target_indices": human_summary["human_target_indices"],
            "human_target_names": human_target_names,
            "human_vote_counts": human_summary["human_vote_counts"],
            "n_missing_human_votes": human_summary["n_missing_human_votes"],

            "warning": game.get("warning", []),
        })

    return pd.DataFrame(rows)


gt_df = build_gt_df(index_data)

print("Ground-truth games:", len(gt_df))
print("Unique game_name:", gt_df["game_name"].nunique())
print()
print("Correct vote type distribution:")
display(gt_df["correct_vote_type"].value_counts(dropna=False).to_frame("n_games"))
print()
print("Number of Werewolves distribution:")
display(gt_df["n_werewolves"].value_counts().sort_index().to_frame("n_games"))

display(gt_df.head())

Ground-truth games: 191
Unique game_name: 191

Correct vote type distribution:


,n_games
correct_vote_type,
player,165
circle,26



Number of Werewolves distribution:


,n_games
n_werewolves,
0,26
1,101
2,64


,game_name,source,session_name,game_key,processed_txt_path,num_players,num_turns,num_unparsed_lines,num_dropped_lines,player_names,...,correct_player_names,correct_vote_type,voting_outcome,human_vote_available,human_vote_type,human_target_indices,human_target_names,human_vote_counts,n_missing_human_votes,warning
0,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#...,Youtube,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#9,Game1,data/processed/lai2023/onuw_transcripts_ready/...,5,112,0,0,"[Justin, Laura, Paul, James, Mitchell]",...,[Justin],player,"[2, 2, 4, 4, 0]",True,player,"[2, 4]","[Paul, Mitchell]","{2: 2, 4: 2, 0: 1}",0,[]
1,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#...,Youtube,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#9,Game2,data/processed/lai2023/onuw_transcripts_ready/...,5,157,0,0,"[Justin, Laura, Paul, James, Mitchell]",...,[Laura],player,"[3, 3, 3, 4, 3]",True,player,[3],[James],"{3: 4, 4: 1}",0,[]
2,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#...,Youtube,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#9,Game3,data/processed/lai2023/onuw_transcripts_ready/...,5,79,0,0,"[Justin, Laura, Paul, James, Mitchell]",...,[Mitchell],player,"[1, 2, 3, 0, 0]",True,player,[0],[Justin],"{1: 1, 2: 1, 3: 1, 0: 2}",0,[]
3,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#...,Youtube,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#9,Game4,data/processed/lai2023/onuw_transcripts_ready/...,5,148,0,0,"[Justin, Laura, Paul, James, Mitchell]",...,[],circle,"[1, 2, 1, 1, 0]",True,player,[1],[Laura],"{1: 3, 2: 1, 0: 1}",0,[No 'Werewolf' role in end roles]
4,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#...,Youtube,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#9,Game5,data/processed/lai2023/onuw_transcripts_ready/...,5,85,0,0,"[Justin, Laura, Paul, James, Mitchell]",...,"[Laura, Paul]",player,"[4, 4, 4, 4, 0]",True,player,[4],[Mitchell],"{4: 4, 0: 1}",0,[]


In [41]:
# -----------------------------
# Discover prompt run directories
# -----------------------------

def strip_prompt_prefix(prompt_family):
    """
    Accepts both:
        'v4'
        'prompt_v4'

    Returns:
        'v4'
    """
    prompt_family = str(prompt_family)

    if prompt_family.startswith("prompt_"):
        return prompt_family[len("prompt_"):]

    return prompt_family


def resolve_model_dir(results_dir, llm):
    """
    Resolve the model directory.

    First tries:
        results_dir / llm

    If it does not exist, it searches for directories whose name contains llm.
    This is useful if the full folder name has extra details.
    """
    model_dir = results_dir / llm

    if model_dir.exists():
        return model_dir

    matches = [
        d for d in results_dir.iterdir()
        if d.is_dir() and llm in d.name
    ]

    if len(matches) == 1:
        print(f"Resolved LLM directory to: {matches[0].name}")
        return matches[0]

    if len(matches) > 1:
        raise ValueError(
            "Multiple matching LLM directories found:\n"
            + "\n".join(str(m) for m in matches)
        )

    raise FileNotFoundError(f"Could not find model directory for llm={llm}")


def discover_prompt_run_dirs(results_dir, llm, prompt_family, include_base=False):
    """
    Finds prompt directories for one prompt family.

    Supports:
        prompt_v4_run_1
        prompt_v4_run_2
        prompt_v4_run_3
        prompt_v4_run1      # accepted just in case
        prompt_v4_t0        # greedy

    Optionally supports:
        prompt_v4           # base directory, ignored by default
    """
    model_dir = resolve_model_dir(results_dir, llm)
    prompt_family_clean = strip_prompt_prefix(prompt_family)

    pattern = re.compile(
        rf"^prompt_{re.escape(prompt_family_clean)}"
        rf"(?:(?:_run_?(\d+))|(?:_t0))?$"
    )

    run_dirs = []

    for d in model_dir.iterdir():
        if not d.is_dir():
            continue

        match = pattern.match(d.name)

        if not match:
            continue

        is_greedy = d.name.endswith("_t0")
        run_number = match.group(1)

        if is_greedy:
            run_dirs.append({
                "prompt_dir": d,
                "prompt_version": d.name,
                "decoding": "greedy",
                "temperature": 0.0,
                "run_number": 0,
                "run_label": "greedy_t0",
                "sort_key": 999,
            })

        elif run_number is not None:
            run_number = int(run_number)

            run_dirs.append({
                "prompt_dir": d,
                "prompt_version": d.name,
                "decoding": "stochastic",
                "temperature": 1.0,
                "run_number": run_number,
                "run_label": f"run_{run_number}",
                "sort_key": run_number,
            })

        else:
            # This is the base folder, e.g. prompt_v4.
            # We usually ignore it because your actual runs are prompt_v4_run_1, etc.
            if include_base:
                run_dirs.append({
                    "prompt_dir": d,
                    "prompt_version": d.name,
                    "decoding": "base",
                    "temperature": np.nan,
                    "run_number": -1,
                    "run_label": "base",
                    "sort_key": -1,
                })

    run_dirs = sorted(run_dirs, key=lambda x: x["sort_key"])

    if not run_dirs:
        raise FileNotFoundError(
            f"No run directories found for prompt family '{prompt_family_clean}' under {model_dir}"
        )

    return run_dirs


prompt_run_dirs = discover_prompt_run_dirs(
    RESULTS_DIR,
    llm=llm,
    prompt_family=prompt_family,
    include_base=False,
)

run_dirs_df = pd.DataFrame([
    {
        "run_label": item["run_label"],
        "decoding": item["decoding"],
        "temperature": item["temperature"],
        "prompt_version": item["prompt_version"],
        "prompt_dir": str(item["prompt_dir"]),
    }
    for item in prompt_run_dirs
])

display(run_dirs_df)

Resolved LLM directory to: unsloth_gemma-4-31B-it-unsloth-bnb-4bit


,run_label,decoding,temperature,prompt_version,prompt_dir
0,run_1,stochastic,1.0,prompt_v4_run_1,C:\Users\annab\Documents\GitHub\masters_thesis...
1,run_2,stochastic,1.0,prompt_v4_run_2,C:\Users\annab\Documents\GitHub\masters_thesis...
2,run_3,stochastic,1.0,prompt_v4_run_3,C:\Users\annab\Documents\GitHub\masters_thesis...
3,greedy_t0,greedy,0.0,prompt_v4_t0,C:\Users\annab\Documents\GitHub\masters_thesis...


In [ ]:
# -----------------------------
# Load and evaluate LLM voting outputs
# -----------------------------

CIRCLE_LABELS = {
    "no werewolf",
}


def first_non_empty(*values):
    """
    Return the first value that is not None and not an empty string.
    """
    for value in values:
        if value is None:
            continue

        if isinstance(value, str) and value.strip() == "":
            continue

        return value

    return None


def normalize_text(x):
    """
    Basic text normalization for votes.
    """
    if x is None:
        return None

    return str(x).strip()


def normalize_for_match(x):
    """
    Stronger normalization for matching labels.
    """
    if x is None:
        return None

    return str(x).strip().lower()


def is_circle_vote_label(vote):
    """
    Detect whether the model selected the no-werewolf / circle option.
    """
    vote_norm = normalize_for_match(vote)

    if vote_norm is None:
        return False

    return vote_norm in CIRCLE_LABELS


def canonicalize_player_vote(chosen_vote, player_names):
    """
    Convert the model's vote to the canonical player name.

    Example:
        "justin" -> "Justin"

    Returns None if the vote is not one of the players.
    """
    chosen_vote = normalize_text(chosen_vote)

    if chosen_vote is None:
        return None

    # Exact match
    if chosen_vote in player_names:
        return chosen_vote

    # Case-insensitive match
    norm_map = {
        str(player).strip().lower(): player
        for player in player_names
    }

    return norm_map.get(chosen_vote.lower(), None)


def build_gt_lookups(gt_df):
    """
    Build lookup tables to match result JSONs to ground truth rows.
    """
    by_full_key = {}
    by_session_game = {}
    by_game_name = {}

    for _, row in gt_df.iterrows():
        full_key = (
            row["source"],
            row["session_name"],
            row["game_key"],
        )

        session_game_key = (
            row["session_name"],
            row["game_key"],
        )

        by_full_key[full_key] = row
        by_game_name[row["game_name"]] = row

        # Store a list just in case the same session/game appears in multiple sources.
        by_session_game.setdefault(session_game_key, []).append(row)

    return by_full_key, by_session_game, by_game_name


gt_by_full_key, gt_by_session_game, gt_by_game_name = build_gt_lookups(gt_df)


def resolve_gt_row(res_data, res_file):
    """
    Find the corresponding ground-truth row for one result JSON.

    Priority:
    1. source + session_name + game_key
    2. session_name + game_key, if unambiguous
    3. JSON filename stem matched against game_name
    """
    source = res_data.get("source")
    session_name = res_data.get("session_name")
    game_key = res_data.get("game_key")

    full_key = (source, session_name, game_key)

    if full_key in gt_by_full_key:
        return gt_by_full_key[full_key]

    session_game_key = (session_name, game_key)

    candidates = gt_by_session_game.get(session_game_key, [])

    if len(candidates) == 1:
        return candidates[0]

    file_stem = Path(res_file).stem

    if file_stem in gt_by_game_name:
        return gt_by_game_name[file_stem]

    return None


def extract_model_vote(res_data):
    """
    Extract the model's chosen vote from the result JSON.

    Your previous notebook used:
        validation["chosen_vote"]
        parsed_output["chosen_vote"]

    This function keeps that behavior but adds a few safe fallbacks.
    """
    validation = res_data.get("validation", {}) or {}
    parsed_output = res_data.get("parsed_output", {}) or {}

    chosen_vote = first_non_empty(
        validation.get("chosen_vote"),
        parsed_output.get("chosen_vote"),
        res_data.get("chosen_vote"),
        res_data.get("vote"),
        res_data.get("final_vote"),
    )

    is_valid = validation.get("is_valid")

    # If is_valid is absent, we infer validity from whether a vote exists.
    if is_valid is None:
        is_valid = chosen_vote is not None

    justification = first_non_empty(
        parsed_output.get("justification"),
        parsed_output.get("reasoning"),
        parsed_output.get("explanation"),
        res_data.get("justification"),
        res_data.get("reasoning"),
        res_data.get("explanation"),
    )

    return chosen_vote, bool(is_valid), justification


def evaluate_one_result_file(res_file, run_info):
    """
    Evaluate one model result JSON against the final roles.
    """
    res_data = load_json(res_file)
    gt_row = resolve_gt_row(res_data, res_file)

    source = res_data.get("source")
    session_name = res_data.get("session_name")
    game_key = res_data.get("game_key")

    chosen_vote_raw, is_valid_parse, justification = extract_model_vote(res_data)

    base_record = {
        "path": str(res_file),

        "run_label": run_info["run_label"],
        "run_number": run_info["run_number"],
        "decoding": run_info["decoding"],
        "temperature": run_info["temperature"],
        "prompt_version": run_info["prompt_version"],

        "source": source,
        "session_name": session_name,
        "game_key": game_key,

        "chosen_vote_raw": chosen_vote_raw,
        "justification": justification,
    }

    if gt_row is None:
        return {
            **base_record,
            "game_name": Path(res_file).stem,
            "status": "missing_ground_truth",
            "chosen_vote": None,
            "canonical_player_vote": None,
            "chosen_player_index": np.nan,
            "voted_player_end_role": None,
            "is_correct": np.nan,
            "is_circle_vote": False,
            "correct_vote_type": None,
            "has_werewolf": np.nan,
            "correct_player_names": None,
            "player_names": None,
            "end_roles": None,
        }

    player_names = gt_row["player_names"]
    end_roles = gt_row["end_roles"]
    correct_player_names = gt_row["correct_player_names"]
    has_werewolf = gt_row["has_werewolf"]
    correct_vote_type = gt_row["correct_vote_type"]

    base_record.update({
        "game_name": gt_row["game_name"],
        "source": gt_row["source"],
        "session_name": gt_row["session_name"],
        "game_key": gt_row["game_key"],
        "player_names": player_names,
        "end_roles": end_roles,
        "has_werewolf": has_werewolf,
        "correct_vote_type": correct_vote_type,
        "correct_player_names": correct_player_names,
    })

    if not is_valid_parse or chosen_vote_raw is None:
        return {
            **base_record,
            "status": "failed_parse",
            "chosen_vote": None,
            "canonical_player_vote": None,
            "chosen_player_index": np.nan,
            "voted_player_end_role": None,
            "is_correct": np.nan,
            "is_circle_vote": False,
        }

    chosen_vote = normalize_text(chosen_vote_raw)
    canonical_player = canonicalize_player_vote(chosen_vote, player_names)

    # Case 1: the model voted for a player.
    if canonical_player is not None:
        chosen_player_index = player_names.index(canonical_player)
        voted_player_end_role = end_roles[chosen_player_index]
        is_correct = voted_player_end_role == "Werewolf"

        return {
            **base_record,
            "status": "player_vote",
            "chosen_vote": chosen_vote,
            "canonical_player_vote": canonical_player,
            "chosen_player_index": chosen_player_index,
            "voted_player_end_role": voted_player_end_role,
            "is_correct": bool(is_correct),
            "is_circle_vote": False,
        }

    # Case 2: the model voted circle / no-werewolf.
    if is_circle_vote_label(chosen_vote):
        is_correct = not has_werewolf

        return {
            **base_record,
            "status": "circle_vote",
            "chosen_vote": "No Werewolf",
            "canonical_player_vote": None,
            "chosen_player_index": np.nan,
            "voted_player_end_role": None,
            "is_correct": bool(is_correct),
            "is_circle_vote": True,
        }

    # Case 3: the model output something that is neither a player nor circle.
    return {
        **base_record,
        "status": "invalid_vote",
        "chosen_vote": chosen_vote,
        "canonical_player_vote": None,
        "chosen_player_index": np.nan,
        "voted_player_end_role": None,
        "is_correct": np.nan,
        "is_circle_vote": False,
    }


def load_all_vote_results(prompt_run_dirs):
    """
    Load all JSON files from all discovered runs.
    """
    rows = []

    for run_info in prompt_run_dirs:
        result_files = sorted(run_info["prompt_dir"].rglob("*.json"))

        print(
            f"{run_info['run_label']:>10} | "
            f"{run_info['decoding']:>10} | "
            f"{run_info['prompt_version']} | "
            f"{len(result_files)} files"
        )

        for res_file in result_files:
            rows.append(evaluate_one_result_file(res_file, run_info))

    return pd.DataFrame(rows)


llm_vote_df = load_all_vote_results(prompt_run_dirs)


# -----------------------------
# Run-level summary
# -----------------------------

def summarize_run(group):
    valid_mask = group["status"].isin(["player_vote", "circle_vote"])
    valid_group = group[valid_mask]

    correct_player_mask = (
        (valid_group["status"] == "player_vote")
        & (valid_group["is_correct"] == True)
    )

    correct_circle_mask = (
        (valid_group["status"] == "circle_vote")
        & (valid_group["is_correct"] == True)
    )

    total_correct = int(valid_group["is_correct"].sum())
    valid_votes_cast = int(valid_mask.sum())

    accuracy = (
        total_correct / valid_votes_cast
        if valid_votes_cast > 0
        else np.nan
    )

    return pd.Series({
        "total_files_found": len(group),
        "total_games_evaluated": int((group["status"] != "missing_ground_truth").sum()),
        "missing_ground_truth": int((group["status"] == "missing_ground_truth").sum()),

        "correct_werewolf_votes": int(correct_player_mask.sum()),
        "correct_circle_votes": int(correct_circle_mask.sum()),
        "total_attempted_circles": int((valid_group["status"] == "circle_vote").sum()),

        "failed_parses": int((group["status"] == "failed_parse").sum()),
        "invalid_votes": int((group["status"] == "invalid_vote").sum()),
        "valid_votes_cast": valid_votes_cast,

        "total_llm_wins": total_correct,
        "accuracy": accuracy,
        "accuracy_percent": accuracy * 100 if pd.notna(accuracy) else np.nan,

        "circle_rate": (
            (valid_group["status"] == "circle_vote").mean()
            if len(valid_group) > 0
            else np.nan
        ),
        "non_circle_accuracy": (
            valid_group.loc[valid_group["status"] == "player_vote", "is_correct"].mean()
            if (valid_group["status"] == "player_vote").any()
            else np.nan
        ),
        "circle_accuracy": (
            valid_group.loc[valid_group["status"] == "circle_vote", "is_correct"].mean()
            if (valid_group["status"] == "circle_vote").any()
            else np.nan
        ),
    })


llm_run_summary_df = (
    llm_vote_df
    .groupby(
        ["prompt_version", "run_label", "run_number", "decoding", "temperature"],
        dropna=False
    )
    .apply(summarize_run)
    .reset_index()
    .sort_values(["decoding", "run_number"])
)

display(llm_run_summary_df)

print()
print("Status counts:")
display(
    llm_vote_df
    .groupby(["run_label", "status"])
    .size()
    .unstack(fill_value=0)
)

     run_1 | stochastic | prompt_v4_run_1 | 191 files
     run_2 | stochastic | prompt_v4_run_2 | 191 files
     run_3 | stochastic | prompt_v4_run_3 | 191 files
 greedy_t0 |     greedy | prompt_v4_t0 | 191 files


C:\Users\annab\AppData\Local\Temp\ipykernel_25296\1523927301.py:419: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_run)


,prompt_version,run_label,run_number,decoding,temperature,total_files_found,total_games_evaluated,missing_ground_truth,correct_werewolf_votes,correct_circle_votes,total_attempted_circles,failed_parses,invalid_votes,valid_votes_cast,total_llm_wins,accuracy,accuracy_percent,circle_rate,non_circle_accuracy,circle_accuracy
3,prompt_v4_t0,greedy_t0,0,greedy,0.0,191.0,191.0,0.0,75.0,9.0,24.0,1.0,0.0,190.0,84.0,0.442105,44.210526,0.126316,0.451807,0.375000
0,prompt_v4_run_1,run_1,1,stochastic,1.0,191.0,191.0,0.0,72.0,8.0,18.0,0.0,0.0,191.0,80.0,0.418848,41.884817,0.094241,0.416185,0.444444
1,prompt_v4_run_2,run_2,2,stochastic,1.0,191.0,191.0,0.0,71.0,11.0,20.0,0.0,0.0,191.0,82.0,0.429319,42.931937,0.104712,0.415205,0.550000
2,prompt_v4_run_3,run_3,3,stochastic,1.0,191.0,191.0,0.0,66.0,11.0,31.0,0.0,0.0,191.0,77.0,0.403141,40.314136,0.162304,0.412500,0.354839



Status counts:


status,circle_vote,failed_parse,player_vote
run_label,,,
greedy_t0,24,1,166
run_1,18,0,173
run_2,20,0,171
run_3,31,0,160


In [35]:
run_summaries = []
all_file_rows = []

for item in prompt_run_dirs:
    prompt_dir = item["prompt_dir"]

    summary, per_file_df = evaluate_llm_prompt_dir(
        prompt_dir,
        ground_truth_lookup_source,
        ground_truth_lookup_fallback,
    )

    summary["run"] = item["run"]
    summary["prompt_dir"] = str(prompt_dir)

    per_file_df["run"] = item["run"]
    per_file_df["prompt_version"] = item["prompt_version"]

    run_summaries.append(summary)
    all_file_rows.append(per_file_df)

llm_run_summary_df = pd.DataFrame(run_summaries).sort_values("run")
llm_file_level_df = pd.concat(all_file_rows, ignore_index=True)

display(llm_run_summary_df)

,prompt_version,total_files_found,total_games_evaluated,missing_ground_truth,correct_werewolf_votes,correct_circle_votes,total_attempted_circles,failed_parses,invalid_votes,valid_votes_cast,total_llm_wins,accuracy,accuracy_percent,run,prompt_dir
0,prompt_v4_run_1,191,191,0,72,8,18,0,0,191,80,0.418848,41.884817,1,C:\Users\annab\Documents\GitHub\masters_thesis...
1,prompt_v4_run_2,191,191,0,71,11,20,0,0,191,82,0.429319,42.931937,2,C:\Users\annab\Documents\GitHub\masters_thesis...
2,prompt_v4_run_3,191,191,0,66,11,31,0,0,191,77,0.403141,40.314136,3,C:\Users\annab\Documents\GitHub\masters_thesis...


In [36]:
import json
import pandas as pd
import numpy as np
from itertools import combinations


def load_run_predictions(prompt_dir, run_label):
    rows = []

    for path in sorted(prompt_dir.rglob("*.json")):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        validation = data.get("validation", {}) or {}
        parsed_output = data.get("parsed_output", {}) or {}

        if not isinstance(parsed_output, dict):
            parsed_output = {}

        chosen_vote = (
            validation.get("chosen_vote")
            or parsed_output.get("chosen_vote")
        )

        is_valid = bool(validation.get("is_valid", False))

        rows.append({
            "run": run_label,
            "source": data.get("source"),
            "session_name": data.get("session_name"),
            "game_key": data.get("game_key"),
            "game_id": (
                str(data.get("source"))
                + " / "
                + str(data.get("session_name"))
                + " / "
                + str(data.get("game_key"))
            ),
            "chosen_vote": str(chosen_vote).strip() if chosen_vote is not None else None,
            "is_valid": is_valid,
            "error_type": data.get("error_type"),
            "path": str(path),
        })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Load predictions from all runs
# ------------------------------------------------------------

all_runs = []

for item in prompt_run_dirs:
    prompt_dir = item["prompt_dir"]
    run_label = item["prompt_version"]   # e.g. prompt_v4, prompt_v4_run_1

    df_run = load_run_predictions(prompt_dir, run_label)
    all_runs.append(df_run)

df_preds = pd.concat(all_runs, ignore_index=True)

print("Files per run:")
display(df_preds.groupby("run").size().reset_index(name="n_files"))


# ------------------------------------------------------------
# Build vote matrix: rows = games, columns = runs
# ------------------------------------------------------------

df_valid = df_preds[df_preds["is_valid"] & df_preds["chosen_vote"].notna()].copy()

vote_matrix = df_valid.pivot_table(
    index="game_id",
    columns="run",
    values="chosen_vote",
    aggfunc="first",
)

run_cols = list(vote_matrix.columns)

print(f"Runs found: {run_cols}")
print(f"Games with at least one valid prediction: {len(vote_matrix)}")


# ------------------------------------------------------------
# Agreement on games valid in all runs
# ------------------------------------------------------------

complete = vote_matrix.dropna(subset=run_cols).copy()

complete["n_unique_votes"] = complete[run_cols].apply(
    lambda row: len(set(row)),
    axis=1,
)

complete["all_same"] = complete["n_unique_votes"] == 1

complete["vote_pattern"] = complete[run_cols].apply(
    lambda row: " | ".join(str(x) for x in row),
    axis=1,
)

n_games = len(complete)
n_same = int(complete["all_same"].sum())
n_diff = n_games - n_same

print("\nOverall agreement:")
print(f"Games valid in all runs: {n_games}")
print(f"Same vote in all runs: {n_same} / {n_games} ({n_same / n_games:.2%})")
print(f"Different vote across runs: {n_diff} / {n_games} ({n_diff / n_games:.2%})")


# ------------------------------------------------------------
# Pairwise agreement
# ------------------------------------------------------------

pairwise = []

for a, b in combinations(run_cols, 2):
    sub = vote_matrix[[a, b]].dropna()
    n = len(sub)
    agree = int((sub[a] == sub[b]).sum())

    pairwise.append({
        "run_a": a,
        "run_b": b,
        "n_common_games": n,
        "n_agree": agree,
        "n_disagree": n - agree,
        "agreement_rate": agree / n if n else np.nan,
    })

pairwise_df = pd.DataFrame(pairwise).sort_values("agreement_rate", ascending=False)

display(pairwise_df)


# ------------------------------------------------------------
# Show games where predictions differ
# ------------------------------------------------------------

disagreements = complete[~complete["all_same"]].copy()
disagreements = disagreements.sort_values(
    ["n_unique_votes", "vote_pattern"],
    ascending=[False, True],
)

print(f"\nDisagreements: {len(disagreements)}")
display(disagreements)

Files per run:


,run,n_files
0,prompt_v4_run_1,191
1,prompt_v4_run_2,191
2,prompt_v4_run_3,191


Runs found: ['prompt_v4_run_1', 'prompt_v4_run_2', 'prompt_v4_run_3']
Games with at least one valid prediction: 191

Overall agreement:
Games valid in all runs: 191
Same vote in all runs: 118 / 191 (61.78%)
Different vote across runs: 73 / 191 (38.22%)


,run_a,run_b,n_common_games,n_agree,n_disagree,agreement_rate
0,prompt_v4_run_1,prompt_v4_run_2,191,144,47,0.753927
1,prompt_v4_run_1,prompt_v4_run_3,191,135,56,0.706806
2,prompt_v4_run_2,prompt_v4_run_3,191,135,56,0.706806



Disagreements: 73


run,prompt_v4_run_1,prompt_v4_run_2,prompt_v4_run_3,n_unique_votes,all_same,vote_pattern
game_id,,,,,,
Youtube / ONE#NIGHT#ULTIMATE#WEREWOLF#9##November#6th#2016 / Game1,Alana,Caitlynn,Mike,3,False,Alana | Caitlynn | Mike
Youtube / ONE#NIGHT#ULTIMATE#WEREWOLF#30##February#17th#2018 / Game4,Alysha,Mitchell,Justin,3,False,Alysha | Mitchell | Justin
Youtube / Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#9 / Game3,James,Laura,No Werewolf,3,False,James | Laura | No Werewolf
Youtube / ONE#NIGHT#ULTIMATE#WEREWOLF#89##November#30th#2019 / Game2,Justin,James,No Werewolf,3,False,Justin | James | No Werewolf
Youtube / ONE#NIGHT#ULTIMATE#WEREWOLF#62##March#23rd#2019 / Game1,Justin,James,Paul,3,False,Justin | James | Paul
...,...,...,...,...,...,...
Youtube / ONE#NIGHT#ULTIMATE#WEREWOLF#44##October#13th#2018 / Game4,Paul,Paul,No Werewolf,2,False,Paul | Paul | No Werewolf
Youtube / Visions#from#2016##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#11 / Game1,Paul,Paul,No Werewolf,2,False,Paul | Paul | No Werewolf
Ego4D / 0c2659db-7bd4-4b37-9b08-4e247befe382 / Game9,Sian,James,James,2,False,Sian | James | James


# Human baseline

In [37]:
def is_missing_human_vote(raw_votes):
    if raw_votes is None:
        return True

    if isinstance(raw_votes, str):
        return raw_votes.strip().upper() in {"NA", "N/A", ""}

    if isinstance(raw_votes, list):
        if len(raw_votes) == 0:
            return True

        normalized = [str(v).strip().upper() for v in raw_votes]
        return any(v in {"NA", "N/A", ""} for v in normalized)

    return False


total_games_in_index = len(index_data)
valid_human_games = 0
human_correct_werewolf_kills = 0
human_correct_circle_votes = 0
human_attempted_circle_votes = 0

for game in index_data:
    raw_votes = game.get("voting_outcome", [])

    if is_missing_human_vote(raw_votes):
        continue

    votes = [int(v) for v in raw_votes]
    end_roles = game.get("end_roles", [])

    valid_human_games += 1

    vote_counts = Counter(votes)

    if not vote_counts:
        continue

    max_votes = max(vote_counts.values())

    # ONUW rule: a player must receive at least 2 votes to be eliminated.
    if max_votes >= 2:
        eliminated_players = [
            p_idx
            for p_idx, count in vote_counts.items()
            if count == max_votes
        ]

        wolf_killed = any(
            end_roles[p_idx] == "Werewolf"
            for p_idx in eliminated_players
        )

        if wolf_killed:
            human_correct_werewolf_kills += 1

    else:
        human_attempted_circle_votes += 1

        if "Werewolf" not in end_roles:
            human_correct_circle_votes += 1


total_human_wins = human_correct_werewolf_kills + human_correct_circle_votes
human_win_rate = (
    total_human_wins / valid_human_games
    if valid_human_games > 0
    else np.nan
)

human_summary_df = pd.DataFrame([{
    "total_games_in_index": total_games_in_index,
    "valid_human_games": valid_human_games,
    "correct_werewolf_kills": human_correct_werewolf_kills,
    "correct_circle_votes": human_correct_circle_votes,
    "total_attempted_circles": human_attempted_circle_votes,
    "total_human_wins": total_human_wins,
    "human_win_rate": human_win_rate,
    "human_win_rate_percent": human_win_rate * 100,
}])

display(human_summary_df)

,total_games_in_index,valid_human_games,correct_werewolf_kills,correct_circle_votes,total_attempted_circles,total_human_wins,human_win_rate,human_win_rate_percent
0,191,171,77,7,12,84,0.491228,49.122807


In [38]:
comparison_with_human_df = llm_run_summary_df[
    [
        "prompt_version",
        "run",
        "valid_votes_cast",
        "total_llm_wins",
        "accuracy",
        "accuracy_percent",
        "failed_parses",
        "invalid_votes",
        "total_attempted_circles",
    ]
].copy()

comparison_with_human_df["human_win_rate"] = human_win_rate
comparison_with_human_df["human_win_rate_percent"] = human_win_rate * 100
comparison_with_human_df["delta_vs_human_percent"] = (
    comparison_with_human_df["accuracy_percent"]
    - comparison_with_human_df["human_win_rate_percent"]
)

display(comparison_with_human_df)

,prompt_version,run,valid_votes_cast,total_llm_wins,accuracy,accuracy_percent,failed_parses,invalid_votes,total_attempted_circles,human_win_rate,human_win_rate_percent,delta_vs_human_percent
0,prompt_v4_run_1,1,191,80,0.418848,41.884817,0,0,18,0.491228,49.122807,-7.237990
1,prompt_v4_run_2,2,191,82,0.429319,42.931937,0,0,20,0.491228,49.122807,-6.190870
2,prompt_v4_run_3,3,191,77,0.403141,40.314136,0,0,31,0.491228,49.122807,-8.808671
